In [1]:
import polars as pl
import numpy as np
from datetime import date
import calendar

# Pesos por día de semana (0=lunes ... 6=domingo)
DAY_WEIGHTS = {0: 1.0, 1: 1.0, 2: 1.0, 3: 1.0, 4: 0.5, 5: 0.0, 6: 0.0}

def get_working_days_with_weights(year: int, month: int) -> tuple[list[date], list[float]]:
    """Devuelve días laborables del mes con sus pesos."""
    num_days = calendar.monthrange(year, month)[1]
    days, weights = [], []
    for d in range(1, num_days + 1):
        day = date(year, month, d)
        w = DAY_WEIGHTS[day.weekday()]
        if w > 0:
            days.append(day)
            weights.append(w)
    return days, weights

def expand_to_daily(df: pl.DataFrame, seed: int = 42) -> pl.DataFrame:
    """
    Expande un DataFrame mensual a diario.
    - Cada trip del registro se convierte en una fila con fecha distinta.
    - Las fechas se sortean con pesos L-J=1.0, V=0.5, finde=0.
    - KM, coste y pallets se mantienen igual en cada fila (sin prorratear).
    """
    rng = np.random.default_rng(seed)
    rows = []

    for row in df.iter_rows(named=True):
        # Parsear mes/año desde "June 2024"
        month_date = date.fromisoformat(
            __import__("datetime").datetime.strptime(row["Fecha"], "%B %Y").strftime("%Y-%m-01")
        )
        year, month = month_date.year, month_date.month
        days, weights = get_working_days_with_weights(year, month)

        if not days:
            continue

        weights_arr = np.array(weights, dtype=float)
        weights_arr /= weights_arr.sum()

        # Número de viajes (redondear por si hay 0.5 trips)
        n_trips = max(1, round(row["trips"]))

        # Sortear n_trips fechas distintas (sin reemplazo si hay suficientes días)
        replace = n_trips > len(days)
        chosen_days = rng.choice(days, size=n_trips, replace=replace, p=weights_arr)

        for trip_date in chosen_days:
            rows.append({**row, "Fecha": trip_date, "trips": 1.0})

    return (
        pl.DataFrame(rows)
        .with_columns(pl.col("Fecha").cast(pl.Date))
        .sort(["Fecha", "Planta", "Destino"])
    )


# --- Cargar datos (ajusta el separador/path según tu fuente) ---
df = pl.read_csv("transporte.csv", separator="\t")

# Renombrar columnas a nombres cortos para trabajar mejor
df = df.rename({
    "SPEC405   [#]   Trips":               "trips",
    "SPEC402   [#]   Drops":               "drops",
    "SPECT1     [#]   Transport days":     "transport_days",
    "SPEC407   [KM]   Paid KM":            "paid_km",
    "SPEC801   [#]   Base Pallets Units":  "pallets",
    "SPEC409   [EUR]   Total Trip Cost":   "total_cost",
})

# Limpiar números europeos (1.429,77 → 1429.77)
for col in ["trips", "transport_days", "paid_km", "pallets", "total_cost"]:
    df = df.with_columns(
        pl.col(col)
        .str.replace_all(r"\.", "")   # quitar separador de miles
        .str.replace(",", ".")         # coma decimal → punto
        .cast(pl.Float64)
    )

df_daily = expand_to_daily(df, seed=42)
print(df_daily.head(20))
print(f"\nFilas originales: {len(df)}  →  Filas diarias: {len(df_daily)}")

# Guardar
df_daily

FileNotFoundError: El sistema no puede encontrar el archivo especificado. (os error 2): transporte.csv